In [1]:
import gymnasium as gym
from gymnasium import spaces
import numpy as np
import pandas as pd
import math
import matplotlib.pyplot as plt
import seaborn as sns
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize
import warnings


In [2]:
class HEMSEnvApplianceScheduling(gym.Env):
    """A HEMS environment that learns to schedule deferrable appliances."""
    metadata = {"render.modes": ["human"]}

    def __init__(self, config):
        super().__init__()
        self.solar_df = pd.read_csv(config["solar_path"], parse_dates=["time"])
        self.solar_df.columns = self.solar_df.columns.str.strip()
        
        # Hardware and simulation parameters from config
        self.solar_panel_area = config["solar_panel_area"]
        self.max_battery = float(config["max_battery_kwh"])
        self.appliances = config["appliances"]
        self.base_load_kw = config["base_load_kw"]
        self.simulation_days = config.get("simulation_days", 365) # Default to 1 year if not specified
        
        self.battery_charge_eff = config.get("battery_charge_eff", 0.95)
        self.battery_discharge_eff = config.get("battery_discharge_eff", 0.95)
        self.battery_deg_cost_per_kwh = config.get("battery_deg_cost_per_kwh", 0.01)
        
        self.n_steps_per_year = len(self.solar_df)
        self.dt_hours = 10 / 60.0
        self.steps_per_day = int(24 / self.dt_hours)
        self.total_simulation_steps = self.simulation_days * self.steps_per_day

        # Gym spaces
        self.action_space = spaces.Discrete(1 + len(self.appliances))
        num_obs_features = 4 + len(self.appliances)
        obs_low = np.array([-1.0] * num_obs_features, dtype=np.float32)
        obs_high = np.array([1.0] * num_obs_features, dtype=np.float32)
        obs_high[0] = 1e6 # solar can be high
        self.observation_space = spaces.Box(obs_low, obs_high, dtype=np.float32)
        
        # Initialize state
        self.time_step = 0
        self.battery_soc = 0.0
        self._reset_appliance_states()
        self.episode_info = []

    def _reset_appliance_states(self):
        self.appliance_states = []
        for app in self.appliances:
            self.appliance_states.append({
                "name": app["name"], "is_running": False,
                "steps_remaining": 0, "task_completed_today": False
            })

    def _compute_solar_output_kw(self, solar_row):
        ghi = pd.to_numeric(solar_row.get("ghi_pyr"), errors='coerce')
        humidity = pd.to_numeric(solar_row.get("relative_humidity"), errors='coerce')
        if pd.isna(ghi) or pd.isna(humidity): return 0.0
        panel_efficiency = 0.18
        weather_factor = max(0.0, 1.0 - (humidity / 100.0))
        pv_kw = (ghi * self.solar_panel_area * panel_efficiency * weather_factor) / 1000.0
        return max(pv_kw, 0.0)

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.time_step = 0
        self.battery_soc = 0.5 * self.max_battery
        self._reset_appliance_states()
        self.episode_info = []
        obs, info = self._get_obs()
        return obs, info

    def _get_obs(self):
        # Use modulo to cycle through the yearly weather data
        idx = self.time_step % self.n_steps_per_year
        solar_row = self.solar_df.iloc[idx]
        
        solar_kw = self._compute_solar_output_kw(solar_row)
        soc_norm = self.battery_soc / self.max_battery if self.max_battery > 0 else 0.0
        
        hour = (self.time_step * 10 // 60) % 24
        hour_rad = (2.0 * math.pi * hour) / 24.0
        
        appliance_statuses = [1.0 if not s["task_completed_today"] else 0.0 for s in self.appliance_states]
        
        obs_list = [solar_kw, soc_norm, math.sin(hour_rad), math.cos(hour_rad)] + appliance_statuses
        return np.array(obs_list, dtype=np.float32), {}

    def _get_grid_price_for_hour(self, hour):
        return 46.85 if 19 <= hour <= 23 else 40.53

    def step(self, action):
        if self.time_step > 0 and self.time_step % self.steps_per_day == 0:
            for state in self.appliance_states:
                state["task_completed_today"] = False

        appliance_load_kw = 0
        for i, state in enumerate(self.appliance_states):
            if state["is_running"]:
                appliance_load_kw += self.appliances[i]["power_kw"]
                state["steps_remaining"] -= 1
                if state["steps_remaining"] == 0:
                    state["is_running"] = False
        
        if action > 0:
            app_idx = action - 1
            if not self.appliance_states[app_idx]["is_running"] and not self.appliance_states[app_idx]["task_completed_today"]:
                self.appliance_states[app_idx]["is_running"] = True
                self.appliance_states[app_idx]["steps_remaining"] = self.appliances[app_idx]["duration_steps"]
                self.appliance_states[app_idx]["task_completed_today"] = True
                appliance_load_kw += self.appliances[app_idx]["power_kw"]

        total_load_kw = self.base_load_kw + appliance_load_kw
        total_load_kwh = total_load_kw * self.dt_hours
        
        idx = self.time_step % self.n_steps_per_year
        solar_row = self.solar_df.iloc[idx]
        solar_kwh_available = self._compute_solar_output_kw(solar_row) * self.dt_hours
        
        grid_energy, solar_supplied, battery_supplied, battery_discharge = 0.0, 0.0, 0.0, 0.0
        remaining_load = total_load_kwh

        solar_supplied = min(solar_kwh_available, remaining_load)
        remaining_load -= solar_supplied
        if self.battery_soc > 0 and remaining_load > 0 and self.battery_discharge_eff > 0:
            discharge = min(self.battery_soc, remaining_load / self.battery_discharge_eff)
            battery_discharge, battery_supplied = discharge, discharge * self.battery_discharge_eff
            self.battery_soc -= discharge
            remaining_load -= battery_supplied
        if remaining_load > 0:
            grid_energy = remaining_load
        if solar_kwh_available > solar_supplied:
            charge = min((solar_kwh_available - solar_supplied) * self.battery_charge_eff, self.max_battery - self.battery_soc)
            self.battery_soc += charge

        hour = (self.time_step * 10 // 60) % 24
        grid_price = self._get_grid_price_for_hour(hour)
        grid_cost = grid_energy * grid_price
        battery_deg_cost = battery_discharge * self.battery_deg_cost_per_kwh
        
        reward = -(grid_cost + battery_deg_cost)
        
        terminated = False
        missed_task_today = False
        if (self.time_step + 1) % self.steps_per_day == 0:
            for state in self.appliance_states:
                if not state["task_completed_today"]:
                    terminated = True
                    missed_task_today = True
                    break
        
        # *** NEW: Use the configured simulation length for termination ***
        if not terminated:
            terminated = self.time_step >= (self.total_simulation_steps - 1)

        info = { "total_cost": grid_cost + battery_deg_cost, "task_missed": missed_task_today }
        self.episode_info.append(info)
        
        self.time_step += 1
        obs, _ = self._get_obs()
        return obs, reward, terminated, False, info

In [5]:
def train_scheduler(env_config, training_timesteps=50000, model_name="ppo_scheduler"):
    print(f"--- Starting Training for Appliance Scheduler ---")
    def make_env():
        return HEMSEnvApplianceScheduling(env_config)

    vec_env = DummyVecEnv([make_env for _ in range(4)])
    vec_env = VecNormalize(vec_env, norm_obs=True, norm_reward=False, clip_obs=10.0)
    
    model = PPO("MlpPolicy", vec_env, policy_kwargs=dict(net_arch=dict(pi=[128, 128], vf=[128, 128])), verbose=0)
    model.learn(total_timesteps=training_timesteps, progress_bar=True)
    
    model.save(model_name)
    vec_env.save(f"{model_name}_vecnormalize.pkl")
    print("--- Training Complete ---")
    return model_name

def evaluate_scheduler(model_name, env_config):
    print(f"\n--- Starting Evaluation for Appliance Scheduler ---")
    model = PPO.load(model_name)
    
    def make_env():
        return HEMSEnvApplianceScheduling(env_config)

    eval_vec_env = DummyVecEnv([make_env])
    eval_vec_env = VecNormalize.load(f"{model_name}_vecnormalize.pkl", eval_vec_env)
    eval_vec_env.training = False
    eval_vec_env.norm_reward = False
    
    all_infos = []
    obs = eval_vec_env.reset()
    done = False
    while not done:
        action, _ = model.predict(obs, deterministic=True)
        obs, _, terminated, info = eval_vec_env.step(action)
        all_infos.append(info[0])
        done = terminated[0]

    return pd.DataFrame(all_infos)

if __name__ == "__main__":
    
    env_config = {
        "solar_path": "solar.csv", 
        "solar_panel_area": 27.8,
        "max_battery_kwh": 7.0, 
        "base_load_kw": 0.3,
        "simulation_days": 30, # <-- Run for one month
        "appliances": [
            {"name": "WaterHeater", "power_kw": 2, "duration_steps": 3},
            {"name": "Dishwasher", "power_kw": 1.8, "duration_steps": 6},
        ]
    }
    
    model_path = train_scheduler(env_config, training_timesteps=150000)
    results = evaluate_scheduler(model_path, env_config)
    
    total_cost = results['total_cost'].sum()
    missed_tasks = results['task_missed'].sum()
    num_days = len(results) / (24 * 6)

    print("\n=== Appliance Scheduling Summary ===")
    print(f"Total Simulation Days: {num_days:.0f}")
    print(f"Total Energy Cost for the month: {total_cost:,.2f} PKR")
    print(f"Total Days with Missed Tasks: {missed_tasks:.0f}")


--- Starting Training for Appliance Scheduler ---


C:\Users\Osama\AppData\Local\Temp\ipykernel_45152\3478791548.py:7: DtypeWarning: Columns (13) have mixed types. Specify dtype option on import or set low_memory=False.
  self.solar_df = pd.read_csv(config["solar_path"], parse_dates=["time"])
C:\Users\Osama\AppData\Local\Temp\ipykernel_45152\3478791548.py:7: DtypeWarning: Columns (13) have mixed types. Specify dtype option on import or set low_memory=False.
  self.solar_df = pd.read_csv(config["solar_path"], parse_dates=["time"])
C:\Users\Osama\AppData\Local\Temp\ipykernel_45152\3478791548.py:7: DtypeWarning: Columns (13) have mixed types. Specify dtype option on import or set low_memory=False.
  self.solar_df = pd.read_csv(config["solar_path"], parse_dates=["time"])
C:\Users\Osama\AppData\Local\Temp\ipykernel_45152\3478791548.py:7: DtypeWarning: Columns (13) have mixed types. Specify dtype option on import or set low_memory=False.
  self.solar_df = pd.read_csv(config["solar_path"], parse_dates=["time"])


c:\Python\Lib\site-packages\rich\live.py:231: UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

--- Training Complete ---

--- Starting Evaluation for Appliance Scheduler ---

=== Appliance Scheduling Summary ===
Total Simulation Days: 1
Total Energy Cost for the month: 0.05 PKR
Total Days with Missed Tasks: 1


C:\Users\Osama\AppData\Local\Temp\ipykernel_45152\3478791548.py:7: DtypeWarning: Columns (13) have mixed types. Specify dtype option on import or set low_memory=False.
  self.solar_df = pd.read_csv(config["solar_path"], parse_dates=["time"])


In [7]:
import pandas as pd
print(pd.read_csv("House16.csv").head())

             Date_Time  Usage_kW  Laundary_kW  Kitchen1_kW  AC_BR_kW  \
0  2018-06-01 00:00:00    4.0747       0.2106       0.0022    0.5674   
1  2018-06-01 00:01:00    4.0643       0.2109       0.0022    0.5675   
2  2018-06-01 00:02:00    4.0065       0.2108       0.0022    0.5675   
3  2018-06-01 00:03:00    3.9424       0.2106       0.0022    0.5686   
4  2018-06-01 00:04:00    3.9665       0.2106       0.0022    0.5687   

   AC_GR_kW  AC_SR_kW  Refrigerator_kW  Kitchen2_kW  AC_BR_kW.1  AC_MBR_kW  
0    0.8622    0.0005           0.0906       0.0006      0.0010     0.0054  
1    0.8545    0.0005           0.0904       0.0006      0.0010     0.0053  
2    0.8562    0.0004           0.0897       0.0006      0.0010     0.0054  
3    0.8560    0.0004           0.0890       0.0006      0.0012     0.0054  
4    0.8594    0.0005           0.0899       0.0006      0.0011     0.0054  
